# nano Beatrix — the relabel read: do the path heads use the path's phases by matching, or by their values?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pillar1j.ipynb)

**The question**: The identity read (`nano_pillar1h.ipynb`) put each path head on fresh random phases: the head holding the lookup lost all of it, and each other path head lost most of its cost too, mostly where the byte's path had not occurred earlier in the row, so where no exact match exists. Random phases take away two things at once: that a path's phases are the same wherever it stands (the exact matches, and keys sharing a path meeting a query at one phase), and the particular values the phase table gave each path, which a model can learn. This notebook separates the two: chosen path heads take the path's phases from a fresh table of the same kind. Equal paths still meet at equal phases, so every exact match is kept; only the values change. Every read is also split by how deep the byte sits in its unit (1, 2, 3, 4, 5 or more bytes), crossed with the repeated-path split. Nothing trains.

| models | sets | on the fresh table and on random phases | reads |
|---|---|---|---|
| the path in blocks 1-3, seeds 1-6 | no block, each of blocks 1, 2, 3 alone, all three | both | 54 |
| the path in every block, seeds 1-6 | no block, each block alone, all six | both | 90 |

With no block changed each read equals the model's stored text read, and the random-phase reads equal the identity read's records, bit for bit: the notebook checks both before it ships anything.

**The cells**: C1 pins the trainer and holds the decision block; C2 writes the held-out file and checks the twelve models, their stored text reads and the identity read's records on [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments); C3 reads each model both ways (a model whose two records are already on the Hub is skipped) and ships the records to `bed/reports/group/` (`<run>_group_relabel_depth.json`, `<run>_group_random_depth.json`); C4 renders each block's cost on the fresh table beside its cost on random phases.

**Before C1**: Runtime > Change runtime type > the G4 card. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on, for the records' upload. Run C1, then Runtime > Restart session and run all. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/ninth_head.py`: `relabel_table`, `relabel_blocks`; `path_rotary.py`: `DepthSplit`; `rotary_scale.py`: `block_read(mode="relabel", depth=True)`; `tests/test_relabel_read.py`).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.98" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 98), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU (the stored reads were taken in bf16 on the card)'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the shipper reads it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out file, the models, the records
REPO = 'AbstractPhil/geolip-rotary-experiments'      # the models, their stored reads and the identity read's records are here; the records ship here
TAIL = lambda L: f'_L{L}_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
PIECE = {'early': 'nhpn16b123', 'every': 'nhpn16'}     # the name's piece: the path in blocks 1-3's ninth heads; in every block's
SERIES = {'early': 'pillar1c', 'every': 'pillar1b'}    # their series
TAG = lambda a, s: f'alhena-both_s{s}{TAIL(24000).replace("_askany_", "_askany_p75d_")}_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_{PIECE[a]}_ahdigit1_{SERIES[a]}'
SETS = {'early': [[], [1], [2], [3], [1, 2, 3]],       # no block, each path block alone, all three (five reads a mode)
        'every': None}                                 # the default: no block, each block alone, all six (eight reads a mode)
MODES = ('relabel', 'random')                          # the fresh table (every exact match kept), then random phases (the identity read's again)
DEPTH = True                                           # every read split by the byte's depth in its unit, crossed with the repeat split
LABEL = {'early': 'the path in blocks 1-3', 'every': 'the path in every block'}
MODELS = [(f'{LABEL[a]}, seed {s}', a, s, TAG(a, s)) for a in ('early', 'every') for s in range(1, 7)]
TEXT_DOCS, TEXT_MB = 128, 16                           # the stored text reads: the trainer's 128 held-out documents in batches of 16
GROUP_DIR = 'bed/reports/group'                        # this notebook's records (series pillar1j) and the identity read's (pillar1h)
TEXT_DIR = 'bed/reports/text'                          # the stored text reads (the parity anchor)
print('decision block:', len(MODELS), 'models |', {a: (len(v) if v else 'no block, each alone, all') for a, v in SETS.items()}, '| modes', MODES,
      '| depth', DEPTH, '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')


In [ ]:
# C2 — THE DATA AND THE MODELS: the held-out head (the trainer's 2,048 reserved fineweb-edu rows), and the twelve models with their
# weights, their ninth heads, their stored text reads and the identity read's records on the Hub.
import json, os, time
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from geolip.nano_beatrix import bed_corpus
from geolip.nano_beatrix import rotary_scale as rs
HELD = bed_corpus.write_heldout(f'{OUT}/bed')
print('held-out head', os.path.basename(HELD), round(os.path.getsize(HELD) / 1e6, 1), 'MB')
api = HfApi(token=HF_TOKEN)
hub = set(api.list_repo_files(REPO))
need = lambda t: (f'bed/weights/{t}/model.safetensors', f'bed/weights/{t}/extra_head.safetensors', f'bed/weights/{t}/scale.json',
                  f'{TEXT_DIR}/{rs.text_name(t)}.json', f'{GROUP_DIR}/{rs.group_name(t, "random")}.json')
missing = [lab for lab, a, s, t in MODELS if not all(f in hub for f in need(t))]
assert not missing, ('models, their stored reads or the identity read records are not on the Hub', missing)
mine = lambda t: [f'{GROUP_DIR}/{rs.group_name(t, m, DEPTH)}.json' for m in MODES]
before = [lab for lab, a, s, t in MODELS if all(f in hub for f in mine(t))]
print(f'the {len(MODELS)} models with their text reads and identity records on {REPO}: {len(MODELS)} of {len(MODELS)}; read here before: {len(before)}')


In [ ]:
# C3 — THE READS: each model fetched from the Hub and read on the fresh table and on random phases (rotary_scale.block_read, modes
# relabel and random, the depth split on), the reads with no block changed checked against the model's stored text read and the
# random reads against the identity read's records, bit for bit; the records shipped to bed/reports/group/ (a background shipper with
# retries). A model whose two records are on the Hub or in OUT/group/ is not read again.
import shutil
lship = rs.Shipper(REPO) if HF_TOKEN else None
os.makedirs(f'{OUT}/group', exist_ok=True)
todo = [m for m in MODELS if not all(f in hub for f in mine(m[3]))]
fetch = lambda path: json.load(open(hf_hub_download(REPO, path, local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
t0 = time.time()
for i, (lab, a, s, tag) in enumerate(todo):
    outs = {m: f'{OUT}/group/{rs.group_name(tag, m, DEPTH)}.json' for m in MODES}
    if not all(os.path.exists(p) for p in outs.values()):   # records made here before a restart: no download, their checks again
        snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    recs = {m: rs.block_read(f'{OUT}/w/bed/weights/{tag}', outs[m], HELD, sets=SETS[a], n_docs=TEXT_DOCS, mb=TEXT_MB, dev='cuda',
                             log=lambda x: None, mode=m, depth=DEPTH) for m in MODES}
    tr = fetch(f'{TEXT_DIR}/{rs.text_name(tag)}.json')
    ident = fetch(f'{GROUP_DIR}/{rs.group_name(tag, "random")}.json')
    by = {m: {tuple(r['off']): r for r in recs[m]['reads']} for m in MODES}
    ib = {tuple(r['off']): r for r in ident['reads']}
    exact = (all(by[m][()]['bits'] == tr['bits'] and by[m][()]['by_repeat'] == tr['by_repeat'] and recs[m].get('mode') == m for m in MODES)
             and set(by['random']) == set(ib) and all(by['random'][o]['bits'] == ib[o]['bits'] and by['random'][o]['by_repeat'] == ib[o]['by_repeat'] for o in ib))
    if not exact:
        for p in outs.values():
            os.replace(p, p + '.bad')                  # kept for the look, never shipped
        raise AssertionError((lab, 'no block changed is not the stored text read, or the random reads are not the identity read: the instrument first',
                              {m: by[m][()]['bits'] for m in MODES}, tr['bits']))
    if lship:
        lship.submit(f'{OUT}/group', GROUP_DIR)
    shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)   # the session's disk holds one model at a time
    el = time.time() - t0
    base = by['random'][()]['bits']
    print(f"{lab}: as trained {base:.4f}; " + ', '.join(f"{'+'.join(map(str, o))}: fresh table {by['relabel'][o]['bits'] - base:+.4f}, random {by['random'][o]['bits'] - base:+.4f}"
          for o in by['random'] if o) + f" | {i + 1} of {len(todo)} read, {el / 60:.1f} min spent, about {el / (i + 1) * (len(todo) - i - 1) / 60:.1f} min left", flush=True)
if lship:
    print('the records shipped' if lship.drain() else 'the records still retrying in the background')


In [ ]:
# C4 — THE TABLE (any time; the records made here or read from the Hub): for each model and path block, the cost on the fresh table
# (what the values carry), the cost on random phases (all the path's phases carry), and their difference (what equality carries).
from IPython.display import Markdown, display

def _rec(local, hub_path):
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))
    try:
        return json.load(open(hf_hub_download(REPO, hub_path, local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
    except Exception:
        return None

for a in ('early', 'every'):
    rows, head = [], None
    for lab, a2, s, tag in MODELS:
        if a2 != a:
            continue
        r = {m: _rec(f'{OUT}/group/{rs.group_name(tag, m, DEPTH)}.json', f'{GROUP_DIR}/{rs.group_name(tag, m, DEPTH)}.json') for m in MODES}
        if any(v is None for v in r.values()):
            continue
        lb = {tuple(x['off']): x['bits'] for x in r['relabel']['reads']}
        rb = {tuple(x['off']): x['bits'] for x in r['random']['reads']}
        sets = [o for o in rb if o and o in lb]
        if head is None:
            head = ['| seed | ' + ' | '.join(f"block{'s' if len(o) > 1 else ''} {'+'.join(map(str, o))}: fresh table / random / equality" for o in sets) + ' |',
                    '|---' * (len(sets) + 1) + '|']
        rows.append(f'| {s} | ' + ' | '.join(f'{lb[o] - lb[()]:+.4f} / {rb[o] - rb[()]:+.4f} / {rb[o] - lb[o]:+.4f}' for o in sets) + ' |')
    if rows:
        display(Markdown(f'**{LABEL[a]}** (held-out bits a byte against the model as trained)\n\n' + '\n'.join(head + rows)))
